This exercise fine tuned a frontier model to get better at predicting the price of products.

In [ ]:
import os
import json
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent.parent))

from dotenv import load_dotenv
from huggingface_hub import login
from openai import OpenAI
from pricer.items import Item
from pricer.evaluator import evaluate

In [2]:
# environment

LITE_MODE = False

load_dotenv(override=True)
hf_token = os.environ['HF_TOKEN']
login(hf_token, add_to_git_credential=True)

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


In [ ]:
username = "ed-donner"
dataset = f"{username}/items_lite" if LITE_MODE else f"{username}/items_full"

train, val, test = Item.from_hub(dataset)

print(f"Loaded {len(train):,} training items, {len(val):,} validation items, {len(test):,} test items")

Loaded 3 test items


In [4]:
openai = OpenAI()

In [44]:
fine_tune_train = train[:100]
fine_tune_validation = val[:50]

In [45]:
def messages_for(item):
    message = f"Estimate the price of this product. Respond with the price, no explanation\n\n{item.summary}"
    return [
        {"role": "user", "content": message},
        {"role": "assistant", "content": f"${item.price:.2f}"}
    ]

In [46]:

def make_jsonl(items):
    result = ""
    for item in items:
        messages = messages_for(item)
        messages_str = json.dumps(messages)
        result += '{"messages": ' + messages_str +'}\n'
    return result.strip()

In [ ]:

def write_jsonl(items, filename):
    with open(filename, "w") as f:
        jsonl = make_jsonl(items)
        f.write(jsonl)

In [47]:
write_jsonl(fine_tune_train, "jsonl/fine_tune_train.jsonl")

In [48]:
write_jsonl(fine_tune_validation, "jsonl/fine_tune_validation.jsonl")

In [39]:
with open("jsonl/fine_tune_train.jsonl", "rb") as f:
    train_file = openai.files.create(file=f, purpose="fine-tune")

In [15]:
with open("jsonl/fine_tune_validation.jsonl", "rb") as f:
    validation_file = openai.files.create(file=f, purpose="fine-tune")

In [ ]:
openai.fine_tuning.jobs.create(
    training_file=train_file.id,
    validation_file=validation_file.id,
    model="gpt-4.1-nano-2025-04-14",
    seed=42,
    hyperparameters={"n_epochs": 3, "batch_size": 8},
    suffix="pricer"
)

FineTuningJob(id='ftjob-GCr7Xk3XzsA242wf0nHSgbo2', created_at=1773184469, error=Error(code=None, message=None, param=None), fine_tuned_model=None, finished_at=None, hyperparameters=Hyperparameters(batch_size=1, learning_rate_multiplier='auto', n_epochs=1), model='gpt-4.1-nano-2025-04-14', object='fine_tuning.job', organization_id='org-fnN7w9whrnItd68BAwoDdeeL', result_files=[], seed=42, status='validating_files', trained_tokens=None, training_file='file-DMU9MchQhC4q9ysdPrcuFC', validation_file='file-YKMP6zpZef5KxF7f5hv4DV', estimated_finish=None, integrations=[], metadata=None, method=Method(type='supervised', dpo=None, reinforcement=None, supervised=SupervisedMethod(hyperparameters=SupervisedHyperparameters(batch_size=1, learning_rate_multiplier='auto', n_epochs=1))), user_provided_suffix='pricer', usage_metrics=None, shared_with_openai=False, eval_id=None, internal_worker_backend=None, internal_peashooter_execution=None, train_experiment_id=None, eval_experiment_id=None)

In [17]:
job_id = openai.fine_tuning.jobs.list(limit=1).data[0].id

In [18]:
openai.fine_tuning.jobs.retrieve(job_id)

FineTuningJob(id='ftjob-GCr7Xk3XzsA242wf0nHSgbo2', created_at=1773184469, error=Error(code=None, message=None, param=None), fine_tuned_model=None, finished_at=None, hyperparameters=Hyperparameters(batch_size=1, learning_rate_multiplier='auto', n_epochs=1), model='gpt-4.1-nano-2025-04-14', object='fine_tuning.job', organization_id='org-fnN7w9whrnItd68BAwoDdeeL', result_files=[], seed=42, status='validating_files', trained_tokens=None, training_file='file-DMU9MchQhC4q9ysdPrcuFC', validation_file='file-YKMP6zpZef5KxF7f5hv4DV', estimated_finish=None, integrations=[], metadata=None, method=Method(type='supervised', dpo=None, reinforcement=None, supervised=SupervisedMethod(hyperparameters=SupervisedHyperparameters(batch_size=1, learning_rate_multiplier='auto', n_epochs=1))), user_provided_suffix='pricer', usage_metrics=None, shared_with_openai=False, eval_id=None, internal_worker_backend=None, internal_peashooter_execution=None, train_experiment_id=None, eval_experiment_id=None)

In [19]:
openai.fine_tuning.jobs.list_events(fine_tuning_job_id=job_id, limit=10).data

[FineTuningJobEvent(id='ftevent-KBbgb0SmCtBVBslU0bVfNjMp', created_at=1773184469, level='info', message='Validating training file: file-DMU9MchQhC4q9ysdPrcuFC and validation file: file-YKMP6zpZef5KxF7f5hv4DV', object='fine_tuning.job.event', data={}, type='message'),
 FineTuningJobEvent(id='ftevent-C7YW4TgFRIVD65mJUXfs93s1', created_at=1773184469, level='info', message='Created fine-tuning job: ftjob-GCr7Xk3XzsA242wf0nHSgbo2', object='fine_tuning.job.event', data={}, type='message')]

In [29]:
fine_tuned_model_name = openai.fine_tuning.jobs.retrieve(job_id).fine_tuned_model

In [31]:
def test_messages_for(item):
    message = f"Estimate the price of this product. Respond with the price, no explanation\n\n{item.summary}"
    return [
        {"role": "user", "content": message},
    ]

In [33]:
def gpt_4__1_nano_fine_tuned(item):
    response = openai.chat.completions.create(
        model=fine_tuned_model_name,
        messages=test_messages_for(item),
        max_tokens=7
    )
    return response.choices[0].message.content

In [34]:
print(test[0].price)
print(gpt_4__1_nano_fine_tuned(test[0]))

219.0
$189.00


In [35]:
evaluate(gpt_4__1_nano_fine_tuned, test)

  0%|          | 0/200 [00:00<?, ?it/s]

$30 $19 $25 $96 $89 $21 $74 $162 $60 $220 $374 $70 $11 $20 $41 $7 $30 $1 $72 $91 $34 $66 $29 $100 $268 $264 $208 $5 $203 $70 $50 $63 $239 $15 $232 $117 $26 $11 $39 $69 $178 $29 $25 $55 $67 $3 $12 $2 $42 $28 $17 $119 $54 $10 $87 $41 $14 $64 $20 $58 $162 $16 $7 $36 $168 $40 $120 $181 $207 $98 $2 $9 $111 $5 $14 $19 $75 $1 $15 $5 $15 $28 $24 $76 $5 $20 $12 $143 $100 $1 $1 $50 $16 $23 $2 $84 $4 $231 $109 $270 $168 $113 $13 $141 $111 $114 $38 $338 $2 $90 $34 $146 $48 $62 $104 $210 $42 $6 $39 $25 $15 $9034 $80 $61 $8 $70 $64 $67 $58 $12 $204 $25 $36 $1 $74 $0 $101 $30 $10 $6 $83 $95 $5 $11 $153 $88 $45 $254 $52 $10 $16 $154 $36 $55 $11 $19 $81 $15 $62 $0 $160 $15 $23 $7 $577 $3 $128 $30 $30 $7 $8 $0 $170 $7 $31 $35 $26 $136 $34 $72 $236 $60 $163 $298 $13 $13 $88 $29 $10 $99 $2 $35 $29 $64 $74 $120 $4 $34 $51 $5 